In [7]:
import pygame as pg
import sys 
import numpy as np

nombre_de_joueur = 2

def BATO(echec,couleur):
    
    #regarder le board
    pieces_a_jouer = []
    for line in board:
        for carre in line:
            if ((carre.piece_color == couleur) and
                (len(carre.show_moves(echec = echec)) > 0)):
                pieces_a_jouer.append(carre)
                
    try: #mettre les try tant qu<il y a des chances que len(mouvement_possible) et len(pieces_a_jouer) peuvent == 0 et qu<ils sont dans des randoms      
        #choisir la piece       
        piece_jouee = pieces_a_jouer[np.random.randint(len(pieces_a_jouer))]

        #bouger la piece
        mouvement_possible = piece_jouee.show_moves(echec = echec)
        mouvement_choisi = mouvement_possible[np.random.randint(len(mouvement_possible))]
        for mouvement in mouvement_possible:
            if board[mouvement[0]][mouvement[1]].occupied:
                mouvement_choisi = mouvement   
                break

            
    except:
        None
    
    
    #envoyer les infos au board
    y1 = piece_jouee.y
    x1 = piece_jouee.x
    y2 = mouvement_choisi[0]
    x2 = mouvement_choisi[1]
    return y1,x1,y2,x2
            
#fonction pour si il y a des pieces qui ne peuvent pas faire de deplacement sinon elles mettent leur propre roi en echec
def pinned(color_turn):
    pinned_pieces = []
    king = False
    for line in board:
        for carre in line:
            carre.pinned = ()
            if ((carre.piece == 'king') and
               (carre.piece_color == color_turn)):
                king = carre
    if king != False:
        for direction in (moves['king'][0]):
            blocking_pieces = []
            for i in range(1,8):
                if ((king.y+direction[0]*i in range(8)) and
                    (king.x+direction[1]*i in range(8))):
                    case_anal = board[king.y+direction[0]*i][king.x+direction[1]*i]
                    if case_anal.piece_color == color_turn:
                        blocking_pieces.append(case_anal)
                    if len(blocking_pieces) > 1:
                        break
                    if ((case_anal.occupied) and
                        (case_anal.piece_color != color_turn) and
                        (len(blocking_pieces) == 1)):
                        if ((moves[case_anal.piece][-1]) and
                            (direction in moves[case_anal.piece][0])):
                            blocking_pieces[0].pinned = ((-direction[0],-direction[1]),direction)
                            break
                        break
                else:
                    break

#appelé seulement lors d'un echec pour eviter de faire des deplacements qui ne sortiront pas le roi de l'echec                         
def deplacement_check(mouvement,piece,piece_color):
    piece_initiale = board[mouvement[0]][mouvement[1]].piece
    occupation_initiale = board[mouvement[0]][mouvement[1]].occupied
    couleur_initiale = board[mouvement[0]][mouvement[1]].piece_color
    board[mouvement[0]][mouvement[1]].piece = piece
    board[mouvement[0]][mouvement[1]].occupied = True
    board[mouvement[0]][mouvement[1]].piece_color = piece_color
    if (not (piece_color in check_check(color = piece_color))):
        board[mouvement[0]][mouvement[1]].piece = piece_initiale
        board[mouvement[0]][mouvement[1]].occupied = occupation_initiale
        board[mouvement[0]][mouvement[1]].piece_color = couleur_initiale
        return True
    else:
        board[mouvement[0]][mouvement[1]].piece = piece_initiale
        board[mouvement[0]][mouvement[1]].occupied = occupation_initiale
        board[mouvement[0]][mouvement[1]].piece_color = couleur_initiale
        return False
    
#fonction pour determiner si une case est a risque de se faire manger
#elle est utilisee pour un roi alors si king = True on evite les iterations infinies
#en definissant les mouvements du roi comme etant tous possible
#Si king = False, on ne l'utilise pas sur un roi alors on peut appeler .show_moves(True)
def danger(pos_case,piece_color,king):

    #Ce if change temporairement la couleur de la case pour la couleur voulue
    if (pos_case[0] in range(8)) and (pos_case[1] in range(8)):
        true_color = board[pos_case[0]][pos_case[1]].piece_color
        board[pos_case[0]][pos_case[1]].piece_color = piece_color

    possible_moves_against_piece = []    
    for line in board:
        for carre in line:
            #2 premiers if seulement si king
            if (carre.occupied and 
               (carre.piece_color != piece_color) and
               (pos_case != (carre.y,carre.x)) and
               (carre.piece != 'king') and
               (king)):
                move = carre.show_moves(moves_and_attacks = False)
                for i in move:
                    possible_moves_against_piece.append(i)   
            #Ce if est necessaire pour eviter les iterations infinies avec la fonction show_moves()
            elif ((carre.piece_color != piece_color) and
               (carre.piece == 'king') and
               (king)):
                for i in range(3):
                    for j in range(3):
                        possible_moves_against_piece.append((carre.y+i-1,carre.x+j-1))
                        
            #dernier if pour les autres pieces ou pour des positions sur le board           
            elif (carre.occupied and 
               (carre.piece_color != piece_color) and
               (pos_case != (carre.y,carre.x)) and
               (carre.piece != 'king') and
               (not king)):
                move = carre.show_moves(moves_and_attacks = True)# la difference entre le premier if et celui ci se trouve ici
                for i in move:
                    possible_moves_against_piece.append(i) 
                                                        
    #La couleur est remise ici                   
    if (pos_case[0] in range(8)) and (pos_case[1] in range(8)):
        board[pos_case[0]][pos_case[1]].piece_color = true_color
        
    if pos_case in possible_moves_against_piece:
        return True
    else:
        return False 
#utilisé dans le cas d'un echec pour determiner si l'échec dont le roi ne peut pas se sortir seul peut être bloqué par une autre pièce
def blockage(pos_king,color_king):
    danger_trajet_echec = []
    for line in board:
        for carre in line:
            if (carre.occupied and 
               (carre.piece_color != color_king) and
               (pos_king in carre.show_moves(moves_and_attacks = False))):
                
                dy = pos_king[0]-carre.y
                dx = pos_king[1]-carre.x
                if abs(dx) == abs(dy):
                    signe_y = dy/abs(dy)
                    signe_x = dx/abs(dx)
                    for i in range(abs(dy)):
                        danger_trajet_echec.append(danger((int(carre.y+signe_y*i),int(carre.x+signe_x*i)),carre.piece_color,False))
                if dx == 0:
                    signe_y = dy/abs(dy)
                    for i in range(abs(dy)):
                        danger_trajet_echec.append(danger((int(carre.y+signe_y*i),carre.x),carre.piece_color,False))  
                if dy == 0:
                    signe_x = dx/abs(dx)
                    for i in range(abs(dx)):
                        danger_trajet_echec.append(danger((carre.y,int(carre.x+signe_x*i)),carre.piece_color,False))
                        
    return any(danger_trajet_echec)

#Pour vérifire la mise en échec des blancs et noirs, ou blanc ou noir selon la variable color
def check_check(color = 'black and white'):
    #trouver les rois et
    #voir si les rois sont en echec
    check_white_king = False
    check_black_king = False
    color_king = ''
    all_white_targets = []
    all_black_targets = []
    for line in board:
        for carre in line:
            if ((carre.piece_color) == 'white'):
                for move in carre.show_moves():
                    all_white_targets.append(move) 
                if (carre.piece == 'king'):
                        mouv_white_king = carre.show_moves(moves_and_attacks = False)
                        pos_white_king = (carre.y,carre.x)
                        if(danger((carre.y,carre.x),carre.piece_color,True)):
                            check_white_king = True 
                            
            if ((carre.piece_color) == 'black'):
                for move in carre.show_moves():
                    all_black_targets.append(move)
                if (carre.piece == 'king'):
                        mouv_black_king = carre.show_moves(moves_and_attacks = False)
                        pos_black_king = (carre.y,carre.x)
                        if(danger((carre.y,carre.x),carre.piece_color,True)):
                            check_black_king = True 


    #voir si les rois sont en echec ou en echec et mat
    if len(all_white_targets) == 0:
        return 'checkmate white'
    if ((check_white_king) and
        ('white' in color)):
        if (not blockage(pos_white_king,'white') and
           (len(mouv_white_king) == 0)):
            return 'checkmate white'
        return 'check white'
    
    if len(all_black_targets) == 0:
        return 'checkmate black'      
    if ((check_black_king) and
        ('black' in color)):
        if (not blockage(pos_black_king,'black') and
           (len(mouv_black_king) == 0)):
            return 'checkmate black'
        return 'check black'
    return 'non'

#Pour faire afficher un string au milieu de l'écran
def affichage_milieu(string):
        font = pg.font.Font(None, 30)
        text = font.render(string, True, (0, 255, 0),(0, 0, 128))
        textRect = text.get_rect()
        textRect.center = (WIDTH/ 2, HEIGHT/ 2)
        screen.blit(text,textRect)

#C'est la seule classe et elle est é.n.o.r.m.e (330 lignes)
#Elle définit l'état d'une case d'échec
class case:
    def __init__(self,y,x):
        self.x = x
        self.y = y
        self.color = (y+x)%2  #1 = black and 0 = white
        self.occupied = False
        self.selected = False
        self.piece_color = None
        self.piece = None
        self.castling = ''
        self.en_passant = False
        self.piece_moved = True
        self.pinned = ()
        if board[y][x] != 0:
            self.occupied = True
            self.piece = board[y][x]
            self.piece_moved = False
            if self.y > 3:
                self.piece_color = "white"
            else:
                self.piece_color = "black"

    
    """Montre les mouvements possibles pour la pièce. Les coordonnées des cases
    où il est possible pour la pièce d'aller sont stockés dans la variable targets
    La première et la plus grosse des 2 seules fonctions de la classe"""
    def show_moves(self,moves_and_attacks = True,echec = False):
        if moves_and_attacks:
            pinned(color_turn)
        move = moves[self.piece]
        targets = []
        
        if not echec:
            """Pawn's special moves"""
            #incluant en passant, premier mouv et pinned()
            if (("pawn" in self.piece) and
                (self.y+1 <= 7) and
                (self.y-1 >= 0)):
                

                """Deplacement de base"""
                if (not board[move[0][0][0] + self.y][move[0][0][1] + self.x].occupied and
                    moves_and_attacks):
                    if len(self.pinned) < 1:
                        targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x))
                    elif (move[0][0][0],move[0][0][1]) in self.pinned:
                        targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x))
                                       
                """Pawn's first move"""
                if (not self.piece_moved and
                    not board[2*move[0][0][0] + self.y][self.x].occupied and#erreur ici quand pion arrive a la fin du board
                    not board[move[0][0][0] + self.y][self.x].occupied and
                    moves_and_attacks):
                    if len(self.pinned) < 1:
                        targets.append((2*move[0][0][0] + self.y,move[0][0][1] + self.x))
                    elif (move[0][0][0],move[0][0][1]) in self.pinned:
                        targets.append((2*move[0][0][0] + self.y,move[0][0][1] + self.x))

                """Pawn's kill moves"""
                #incluant le en_passant
                if self.x+1 <= 7:
                    if not moves_and_attacks:
                        if len(self.pinned) < 1:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x+1))
                        elif (move[0][0][0],move[0][0][1]+1) in self.pinned:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x+1))                   
                    if (board[self.y + move[0][0][0]][self.x+1].occupied and 
                        board[self.y + move[0][0][0]][self.x+1].piece_color != self.piece_color):
                        if len(self.pinned) < 1:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x+1))
                        elif (move[0][0][0],move[0][0][1]+1) in self.pinned:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x+1))
                    if board[self.y + move[0][0][0]][self.x+1].en_passant:
                        if len(self.pinned) < 1:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x+1))
                        elif (move[0][0][0],move[0][0][1]+1) in self.pinned:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x+1))

                if self.x-1 >= 0:
                    if not moves_and_attacks:
                        if len(self.pinned) < 1:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x-1))
                        elif (move[0][0][0],move[0][0][1]-1) in self.pinned:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x-1))                
                    if (board[self.y + move[0][0][0]][self.x-1].occupied and
                        board[self.y + move[0][0][0]][self.x-1].piece_color != self.piece_color):
                        if len(self.pinned) < 1:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x-1))
                        elif (move[0][0][0],move[0][0][1]-1) in self.pinned:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x-1))
                    if board[self.y + move[0][0][0]][self.x-1].en_passant:    
                        if len(self.pinned) < 1:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x-1))
                        elif (move[0][0][0],move[0][0][1]-1) in self.pinned:
                            targets.append((move[0][0][0] + self.y,move[0][0][1] + self.x-1))

                return(targets)       

            #Le roi peut bouger seulement si il n'est pas en danger et il peut faire le roque
            #Le roi ne peut pas être pinned (duh)
            if "king" in self.piece:
                #pour verifier les attaques comme si le roi n'etait pas la
                self.occupied = False
                color = self.piece_color
                self.piece_color = None
                self.piece = None

                #castling droit
                if ((moves_and_attacks) and
                    (not self.piece_moved) and
                    (not board[self.y][7].piece_moved) and
                    (all(not board[self.y][self.x+x].occupied for x in (1,2))) and
                    (all(not danger((self.y,self.x+x),color,True) for x in (1,2)))):#on met true pour eviter les iterations infinies
                    targets.append((self.y,self.x+2))
                    self.castling += 'droit'

                #castling gauche
                if ((moves_and_attacks) and
                    (not self.piece_moved) and
                    (not board[self.y][0].piece_moved) and
                    (all(not board[self.y][self.x-x].occupied for x in (1,2,3))) and
                    (all(not danger((self.y,self.x-x),color,True) for x in (1,2)))):#on met true pour eviter les iterations infinies
                    targets.append((self.y,self.x-2)) 
                    self.castling += 'gauche'

                #mouvements seulement si aucun danger    
                for i in move[0]:
                    dest = (i[0] + self.y,i[1] + self.x)
                    if ((dest[0] in range(8)) and 
                       (dest[1] in range(8)) and 
                       (not danger((dest),color,True)) and#le roi peut bouger seulement si la ou il va il n'y a pas de danger
                       (board[dest[0]][dest[1]].piece_color != color)):
                        targets.append(dest)
                #on remet le roi a sa place
                self.occupied = True
                self.piece_color = color
                self.piece = 'king'

                return(targets)

            #deplacement des autres pieces sauf le pawn et le king
            for i in move[0]:
                dist = 1
                for _ in range(7):  #Très professionel
                    dest = (dist*i[0] + self.y,dist*i[1] + self.x)

                    if  dest[0] in range(8) and dest[1] in range(8):
                        if board[dest[0]][dest[1]].piece_color == self.piece_color:
                            break
                        if len(self.pinned) < 1:
                            targets.append(dest)
                        elif i in self.pinned:
                            targets.append(dest)
                        if board[dest[0]][dest[1]].occupied:
                            break

                    dist += 1
                    if not move[-1]:
                        break
        #Echec = True seulement si la couleur est en échec
        #Dans ce cas on restreint les mouvements des pièces pour jouer seulement ce qui sort de l'échec
        if echec:
            #On enleve la piece pour voir ce quelle peut faire pour les échecs
            #La fonction deplacement_check() met la piece ou elle veut se déplacer, regarde si on est encore en état d'échec
            #, si oui elle retourne False et, dans tous les cas, la réenlève
            piece = self.piece
            piece_color = self.piece_color
            board[self.y][self.x].piece = None
            board[self.y][self.x].occupied = False
            board[self.y][self.x].piece_color = None
            """Pawn's special moves"""
            if (("pawn" in piece) and
                (self.y <= 7) and
                (self.y-1 >= 0)):

                """Deplacement de base"""
                if ((not board[move[0][0][0] + self.y][move[0][0][1] + self.x].occupied) and
                    (moves_and_attacks)):
                    mouvement = (move[0][0][0] + self.y,move[0][0][1] + self.x)
                    if deplacement_check(mouvement,piece,piece_color):
                        targets.append(mouvement)

                """Pawn's first move"""
                
                if (not self.piece_moved and
                    not board[2*move[0][0][0] + self.y][2*move[0][0][1] + self.x].occupied and
                    moves_and_attacks):
                    mouvement = (2*move[0][0][0] + self.y,self.x)
                    if deplacement_check(mouvement,piece,piece_color):
                        targets.append(mouvement)


                """Pawn's kill moves"""
                #incluant le en_passant
                if self.x+1 <= 7:                   
                    if (board[self.y + move[0][0][0]][self.x+1].occupied and 
                        board[self.y + move[0][0][0]][self.x+1].piece_color != self.piece_color):
                        mouvement = (self.y + move[0][0][0],self.x+1)
                        if deplacement_check(mouvement,piece,piece_color):
                            targets.append(mouvement)
                    if board[self.y + move[0][0][0]][self.x+1].en_passant:
                        mouvement = (self.y + move[0][0][0],self.x+1)
                        if deplacement_check(mouvement,piece,piece_color):
                            targets.append(mouvement)

                if self.x-1 >= 0:               
                    if (board[self.y + move[0][0][0]][self.x-1].occupied and
                        board[self.y + move[0][0][0]][self.x-1].piece_color != self.piece_color):
                        mouvement = (self.y + move[0][0][0],self.x-1)
                        if deplacement_check(mouvement,piece,piece_color):
                            targets.append(mouvement)
                    if board[self.y + move[0][0][0]][self.x-1].en_passant:    
                        mouvement = (self.y + move[0][0][0],self.x-1)
                        if deplacement_check(mouvement,piece,piece_color):
                            targets.append(mouvement)
                #on remet la piece a sa place avant chaque return
                board[self.y][self.x].piece = piece
                board[self.y][self.x].occupied = True
                board[self.y][self.x].piece_color = piece_color
                return(targets)       

            #Le roi peut bouger seulement si il n'est pas en danger et il peut faire le roque
            if "king" in piece:

                #castling droit
                if ((moves_and_attacks) and
                    (not self.piece_moved) and
                    (not board[self.y][7].piece_moved) and
                    (all(not board[self.y][self.x+x].occupied for x in (1,2))) and
                    (all(not danger((self.y,self.x+x),piece_color,True) for x in (1,2)))):#on met true pour eviter les iterations infinies
                    mouvement = (self.y,self.x+2)
                    if deplacement_check(mouvement,piece,piece_color):
                        targets.append(mouvement)
                        self.castling += 'droit'
                        

                #castling gauche
                if ((moves_and_attacks) and
                    (not self.piece_moved) and
                    (not board[self.y][0].piece_moved) and
                    (all(not board[self.y][self.x-x].occupied for x in (1,2,3))) and
                    (all(not danger((self.y,self.x-x),piece_color,True) for x in (1,2)))):#on met true pour eviter les iterations infinies
                    mouvement = (self.y,self.x-2)
                    if deplacement_check(mouvement,piece,piece_color):
                        targets.append(mouvement)
                        self.castling += 'gauche'

                #mouvements seulement si aucun danger    
                for i in move[0]:
                    dest = (i[0] + self.y,i[1] + self.x)
                    if ((dest[0] in range(8)) and 
                       (dest[1] in range(8)) and 
                       (not danger((dest),piece_color,True)) and#le roi peut bouger seulement si la ou il va il n'y a pas de danger
                       (board[dest[0]][dest[1]].piece_color != piece_color)):
                        mouvement = dest
                        if deplacement_check(mouvement,piece,piece_color):
                            targets.append(mouvement)
                #on remet la piece a sa place
                board[self.y][self.x].piece = piece
                board[self.y][self.x].occupied = True
                board[self.y][self.x].piece_color = piece_color
                return(targets)

            #deplacement des autres pieces sauf le pawn et le king
            for i in move[0]:
                dist = 1
                for _ in range(7):  #Très professionel
                    dest = (dist*i[0] + self.y,dist*i[1] + self.x)

                    if  dest[0] in range(8) and dest[1] in range(8):
                        if board[dest[0]][dest[1]].piece_color == piece_color:
                            break
                        mouvement = dest
                        if deplacement_check(mouvement,piece,piece_color):
                            targets.append(mouvement)
                        if board[dest[0]][dest[1]].occupied:
                            break

                    dist += 1
                    if not move[-1]:
                        break
            #on remet la piece a sa place
            board[self.y][self.x].piece = piece
            board[self.y][self.x].occupied = True
            board[self.y][self.x].piece_color = piece_color
        return(targets)
    
    """Fonction appelée pour bouger une pièce. La case b est un objet qui représente la case de destination, pas des coordonnées"""
    def move(self,b):
        
        if b.occupied:
            last_move = (self.y,self.x,b.y,b.x,b.piece,color_turn,self.piece,None)
        else:
            last_move = (self.y,self.x,b.y,b.x,None,color_turn,self.piece,None)    

            
        #gestion des en passants
        if b.en_passant:
            last_move = (self.y,self.x,b.y,b.x,board[self.y][b.x].piece,color_turn,self.piece,'en passant') 
            board[self.y][b.x].piece = None
            board[self.y][b.x].occupied = False
            board[self.y][b.x].piece_color = None
            board[self.y][b.x].piece_moved = True
            
        for line in board:
            for case in line:
                if case.en_passant:
                    case.en_passant = False
                    
        #mouvements du en passant
        if (((self.piece == 'bpawn') or
            (self.piece == 'wpawn')) and
            (abs(self.y - b.y) == 2)):
            board[int(self.y + (b.y-self.y)/2)][b.x].en_passant = True
            
        #mouvement des roques 
        #on s'occupe seulement du mouvement des tours dans cette partie-ci
        if (('droit' in self.castling) and
            ((self.x - b.x) == -2)):
            board[self.y][5].piece = 'rook'
            board[self.y][5].occupied = True
            board[self.y][5].piece_color = self.piece_color
            board[self.y][5].piece_moved = True   
            
            board[self.y][7].piece = None
            board[self.y][7].occupied = False
            board[self.y][7].piece_color = None
            board[self.y][7].piece_moved = True
            last_move = (self.y,self.x,b.y,b.x,board[self.y][b.x].piece,color_turn,self.piece,'castling droit')
        if (('gauche' in self.castling) and
            ((self.x - b.x) == 2)):
            
            board[self.y][3].piece = 'rook'
            board[self.y][3].occupied = True
            board[self.y][3].piece_color = self.piece_color
            board[self.y][3].piece_moved = True   
            
            board[self.y][0].piece = None
            board[self.y][0].occupied = False
            board[self.y][0].piece_color = None
            board[self.y][0].piece_moved = True   
            last_move = (self.y,self.x,b.y,b.x,board[self.y][b.x].piece,color_turn,self.piece,'castling gauche')
        
        #mouvement décidé des pièces
        b.occupied = True
        b.piece = self.piece
        b.piece_color = self.piece_color
        b.piece_moved = True
        if "pawn" in b.piece and b.y == 7:
            b.piece = "queen"
            last_move = (self.y,self.x,b.y,b.x,board[self.y][b.x].piece,color_turn,self.piece,'reine') 
        if "pawn" in b.piece and b.y == 0:
            b.piece = "queen" 
            last_move = (self.y,self.x,b.y,b.x,board[self.y][b.x].piece,color_turn,self.piece,'reine') 

        self.piece = None
        self.occupied = False
        self.piece_color = None
        self.piece_moved = True
        self.castling = ''
        
        all_last_move.append(last_move)
        return last_move



        
            

                

"""Les différents mouvements possibles pour chaque pièce. La dernière donnée est pour indiquer si le mouvement de la pièce est 'infini'
Par exemple, le roi et le cheval peuvent seulement faire un mouvement. Mais pour la tour, la reine et le fou, ils peuvent aller aussi loin
qu'ils le veulent dans une direction."""
moves = {   "rook" : ([(0,1),(1,0),(-1,0),(0,-1)],True),
            "knight" : ([(2,1),(2,-1),(-2,1),(-2,-1),(1,2),(1,-2),(-1,2),(-1,-2)],False),
            "bishop" : ([(1,1),(-1,-1),(-1,1),(1,-1)],True),
            "king" : ([(0,1),(1,0),(-1,0),(0,-1),(1,1),(-1,-1),(-1,1),(1,-1)],False),
            "queen" : ([(0,1),(1,0),(-1,0),(0,-1),(1,1),(-1,-1),(-1,1),(1,-1)],True),
            "wpawn" : ([(-1,0)],False),
            "bpawn" : ([(1,0)],False)}


board =   [["rook","knight","bishop","queen","king","bishop","knight","rook"],
        ["bpawn"]*8,
        [0]*8,
        [0]*8,
        [0]*8,
        [0]*8,
        ["wpawn"]*8,
        ["rook","knight","bishop","queen","king","bishop","knight","rook"]]

def init_boardgame():

    """Initialisation du board en le remplissant d'objets."""
    for y in range(len(board)):
        for x in range(len(board[y])):
            board[y][x] = case(y,x)
            
init_boardgame()

"""Paramètres pour l'affichage, initialisation pygame et variables."""
WIDTH,HEIGHT = (800,800)
cs = HEIGHT/(len(board)) #Case Size

pg.init()
screen = pg.display.set_mode((WIDTH,HEIGHT))
pg.display.set_caption("PyChess")

imagesW = { "rook"   :  (pg.image.load("chess_piece_2_white_rook.png").convert()),
            "knight" :  (pg.image.load("chess_piece_2_white_knight.png").convert()),
            "bishop" :  (pg.image.load("chess_piece_2_white_bishop.png").convert()),
            "king"   :  (pg.image.load("chess_piece_2_white_king.png").convert()),
            "queen"  :  (pg.image.load("chess_piece_2_white_queen.png").convert()),
            "wpawn"  :  (pg.image.load("chess_piece_2_white_pawn.png").convert())}

imagesB = { "rook"   :  (pg.image.load("chess_piece_2_black_rook.png").convert()),
            "knight" :  (pg.image.load("chess_piece_2_black_knight.png").convert()),
            "bishop" :  (pg.image.load("chess_piece_2_black_bishop.png").convert()),
            "king"   :  (pg.image.load("chess_piece_2_black_king.png").convert()),
            "queen"  :  (pg.image.load("chess_piece_2_black_queen.png").convert()),
            "bpawn"  :  (pg.image.load("chess_piece_2_black_pawn.png").convert())}


targets = []       #Les cases où la pièce sélectionnée peut se déplacer
selected = []      #La pièce sélectionnée
condition_echec = 'non' # 5 valeurs possible: non, check white, check black, checkmate white et checkmate black
echec_blanc = False #retourne True si le mot check se trouve dans condition_echec
echec_noir = False
echec = False
timer = 0
timer_pause = 0
pause = True
key_pause = False
all_last_move = []
all_future_move = []
last_move = False


"""La couleur à qui s'est le tour de jouer."""
color_turn = "white"


"""Main loop pour faire fonctionner le jeu"""
while True:
    screen.fill((51,51,51))
    screen.set_alpha(100)
    
    """On va chercher les input (les clics, les touches touchées et la position de la souris)"""
    mouse = pg.mouse.get_pos()
    mouse_state = pg.mouse.get_pressed()
    key = pg.key.get_pressed()
    
    if len(all_future_move) == 0:
        #jeu humain vs humain
        if nombre_de_joueur == 2:
            if mouse_state[0]:
                """Coordonnées de la souris"""
                x = int(mouse[0]/cs)
                y = int(mouse[1]/cs)

                # on passe ce if si on clique dans une case ou le mouvement est permis (targets)
                if (y,x) in targets and len(selected) > 1:

                    last_move = board[selected[0]][selected[1]].move(board[y][x])
                    condition_echec = check_check()
                    echec = (True if 'check' in condition_echec else False)
                    all_targets = []


                    selected = []
                    if color_turn == "white":
                        color_turn = "black"
                    else:
                        color_turn = "white"
                #on passe ce if si on clique sur une de nos pièces
                elif board[y][x].occupied and board[y][x].piece_color == color_turn:
                    selected = (y,x)
                    targets = board[y][x].show_moves(echec = echec)

                else:
                    targets = []
                    selected = []

        #Jeu humain(blanc) vs robot(noir)
        if nombre_de_joueur == 1:  
            #humain
            if color_turn == "white":
                if mouse_state[0]:
                    """Coordonnées de la souris"""
                    x = int(mouse[0]/cs)
                    y = int(mouse[1]/cs)

                    # on passe ce if si on clique dans une case ou le mouvement est permis (targets)
                    if (y,x) in targets and len(selected) > 1:


                        last_move = board[selected[0]][selected[1]].move(board[y][x])
                        condition_echec = check_check()
                        echec_noir = (True if 'check' in condition_echec else False)
                        selected = []
                        targets = []
                        color_turn = "black"
                    #on passe ce if si on clique sur une de nos pièces
                    elif board[y][x].occupied and board[y][x].piece_color == color_turn:
                        selected = (y,x)
                        targets = board[y][x].show_moves(echec = echec_blanc)

                    else:
                        targets = []
                        selected = []
            #robot            
            timer_pause += 1
            if timer_pause > 500:
                pause = False
                timer_pause = 0
            else:
                pause = True
            if ((color_turn == "black") and
                (not pause) and
                (not key_pause)):             
                y1,x1,y2,x2 = BATO(echec_noir,'black')
                last_move = board[y1][x1].move(board[y2][x2])

                condition_echec = check_check()
                echec_blanc = (True if 'check' in condition_echec else False)  

                color_turn = "white"


        #Jeu robot vs robot
        if nombre_de_joueur == 0:    

            timer_pause += 1
            if timer_pause > 200:
                pause = False
                timer_pause = 0
            else:
                pause = True
            if ((not pause) and
                (not key_pause)):
                y1,x1,y2,x2 = BATO(echec,color_turn)
                last_move = board[y1][x1].move(board[y2][x2])

                condition_echec = check_check()
                echec = (True if 'check' in condition_echec else False)  

                if color_turn == "white":
                    color_turn = "black"
                else:
                    color_turn = "white"



    """Dessin du tableau"""
    for line in board:
        for carre in line:
            # On se dessine a chaque fois pour ecraser ce qui est superflu
            #Ca pourrait causer un trop grand nombre d'objets à gérer vu qu'on écrase et on enlève pas?

            if not carre.color:
                case_color = (200, 200, 200)
                rect = pg.Rect(carre.x*cs,carre.y*cs,cs,cs)
                pg.draw.rect(screen,case_color,rect)
            if carre.color:
                case_color = (50, 50, 50)
                rect = pg.Rect(carre.x*cs,carre.y*cs,cs,cs)
                pg.draw.rect(screen,case_color,rect)
            if (carre.y,carre.x) in targets:
                case_color = (0,0,100)
                rect = pg.Rect(carre.x*cs,carre.y*cs,cs,cs)
                pg.draw.rect(screen,case_color,rect)
            ### pour les tests de la fonction pinned()###
            if carre.pinned:
                case_color = (100,0,0)
                rect = pg.Rect(carre.x*cs,carre.y*cs,cs,cs)
                pg.draw.rect(screen,case_color,rect)   
            ### fin du test ###    
            
            #Si la case est occupée on dessine dessus
            if carre.occupied:   
                if carre.piece_color == "white":
                    img = imagesW[carre.piece]
                    screen.blit(img,(int((carre.x+.3)*cs),int((carre.y+.3)*cs)))
                elif carre.piece_color == "black":
                    img = imagesB[carre.piece]
                    screen.blit(img,(int((carre.x+.3)*cs),int((carre.y+.3)*cs)))

    #dépendammant des conditions d'échec on écrit des trucs sur le board
    # On ne peut pas cliquer sur le board a cause du timer
    if condition_echec == 'check white':
        affichage_milieu('Roi blanc en échec')
        targets = []
        timer += 1
        if timer > 100:
            condition_echec = 'non'
            timer = 0
    if condition_echec == 'checkmate white':
        affichage_milieu('Roi blanc en échec et mat')
        targets = []
        timer += 1
        key_pause = True
        if timer > 100:
            condition_echec = 'non'
            timer = 0

    if condition_echec == 'check black':
        affichage_milieu('Roi noir en échec')
        targets = []
        timer += 1
        if timer > 100:
            condition_echec = 'non'
            timer = 0
    if condition_echec == 'checkmate black':
        affichage_milieu('Roi noir en echec et mat')
        targets = []
        timer += 1
        key_pause = True
        if timer > 100:
            condition_echec = 'non'
            timer = 0
            

                    
    pg.display.flip()#je sais pas a quoi ca sert


    
    """On regarde si le joueur veut quitter"""
    for event in pg.event.get():
        if event.type == pg.QUIT or key[pg.K_ESCAPE]:
            pg.quit()
            sys.exit()
        if ( event.type == pg.KEYDOWN ):
            if event.key == pg.K_r :
                board =   [["rook","knight","bishop","queen","king","bishop","knight","rook"],
                        ["bpawn"]*8,
                        [0]*8,
                        [0]*8,
                        [0]*8,
                        [0]*8,
                        ["wpawn"]*8,
                        ["rook","knight","bishop","queen","king","bishop","knight","rook"]]
                init_boardgame()
                color_turn = "white"
                all_last_move = []
                all_future_move = []
                last_move = False
            if event.key == pg.K_p :
                if key_pause == False:
                    key_pause = True
                else:
                    key_pause = False
                
            if event.key == pg.K_LEFT :
                try:
                    last_move = all_last_move[-1]
                    all_future_move.append(last_move)
                    all_last_move.pop()
                except:
                    break
             
                if last_move[-1] == 'en passant':
                    board[last_move[0]][last_move[1]].occupied = True
                    board[last_move[0]][last_move[1]].piece = last_move[6]
                    board[last_move[0]][last_move[1]].piece_color = last_move[5]

                    board[last_move[2]][last_move[3]].occupied = False
                    board[last_move[2]][last_move[3]].piece = None
                    board[last_move[2]][last_move[3]].piece_color = None
                    
                    board[last_move[0]][last_move[3]].occupied = True
                    board[last_move[0]][last_move[3]].piece = last_move[4]
                    if last_move[5] == 'black':
                        board[last_move[0]][last_move[3]].piece_color = 'white' 
                    else :
                        board[last_move[0]][last_move[3]].piece_color = 'black'
                
                elif last_move[-1] == 'castling droit':
                    #pour le roi
                    board[last_move[0]][last_move[1]].occupied = True
                    board[last_move[0]][last_move[1]].piece = last_move[6]
                    board[last_move[0]][last_move[1]].piece_color = last_move[5]
                    board[last_move[0]][last_move[1]].piece_moved = True

                    board[last_move[2]][last_move[3]].occupied = False
                    board[last_move[2]][last_move[3]].piece = None
                    board[last_move[2]][last_move[3]].piece_color = None
                    board[last_move[2]][last_move[3]].piece_moved = True  
                    
                    #pour la tour
                    board[last_move[2]][last_move[3]-1].occupied = False
                    board[last_move[2]][last_move[3]-1].piece = None
                    board[last_move[2]][last_move[3]-1].piece_color = None
                    board[last_move[2]][last_move[3]-1].piece_moved = True  

                    board[last_move[2]][7].occupied = True
                    board[last_move[2]][7].piece = 'rook'
                    board[last_move[2]][7].piece_color = last_move[5]
                    board[last_move[2]][7].piece_moved = True       
                elif last_move[-1] == 'castling gauche':
                    #pour le roi
                    board[last_move[0]][last_move[1]].occupied = True
                    board[last_move[0]][last_move[1]].piece = last_move[6]
                    board[last_move[0]][last_move[1]].piece_color = last_move[5]
                    board[last_move[0]][last_move[1]].piece_moved = True

                    board[last_move[2]][last_move[3]].occupied = False
                    board[last_move[2]][last_move[3]].piece = None
                    board[last_move[2]][last_move[3]].piece_color = None
                    board[last_move[2]][last_move[3]].piece_moved = True  
                    
                    #pour la tour
                    board[last_move[2]][last_move[3]+1].occupied = False
                    board[last_move[2]][last_move[3]+1].piece = None
                    board[last_move[2]][last_move[3]+1].piece_color = None
                    board[last_move[2]][last_move[3]+1].piece_moved = True  

                    board[last_move[2]][0].occupied = True
                    board[last_move[2]][0].piece = 'rook'
                    board[last_move[2]][0].piece_color = last_move[5]
                    board[last_move[2]][0].piece_moved = True                           
                else :
                    board[last_move[0]][last_move[1]].occupied = True
                    board[last_move[0]][last_move[1]].piece = last_move[6]
                    board[last_move[0]][last_move[1]].piece_color = last_move[5]
                    board[last_move[0]][last_move[1]].piece_moved = True

                    #remettre les kills
                    board[last_move[2]][last_move[3]].piece = last_move[4]
                    if ((last_move[4] != None) and
                       (last_move[5] == 'black')):
                        board[last_move[2]][last_move[3]].piece_color = 'white' 
                        board[last_move[2]][last_move[3]].occupied = True
                    elif ((last_move[4] != None) and
                         (last_move[5] == 'white')) :
                        board[last_move[2]][last_move[3]].piece_color = 'black'
                        board[last_move[2]][last_move[3]].occupied = True
                    else:
                        board[last_move[2]][last_move[3]].piece_color = None
                        board[last_move[2]][last_move[3]].occupied = False
                    board[last_move[2]][last_move[3]].piece_moved = True
                    
                
                
            if event.key == pg.K_RIGHT :
                try:
                    future_move = all_future_move[-1]
                    all_last_move.append(future_move)
                    all_future_move.pop()
                except:
                    break

                
                if future_move[-1] == 'en passant':
                    board[future_move[2]][future_move[3]].occupied = True
                    board[future_move[2]][future_move[3]].piece = future_move[6]
                    board[future_move[2]][future_move[3]].piece_color = future_move[5]

                    board[future_move[0]][future_move[1]].occupied = False
                    board[future_move[0]][future_move[1]].piece = None
                    board[future_move[0]][future_move[1]].piece_color = None
                    
                    board[future_move[0]][future_move[3]].occupied = False
                    board[future_move[0]][future_move[3]].piece = None
                    board[future_move[0]][future_move[3]].piece_color = None
                elif future_move[-1] == 'reine':
                    board[future_move[2]][future_move[3]].occupied = True
                    board[future_move[2]][future_move[3]].piece = 'queen'
                    board[future_move[2]][future_move[3]].piece_color = future_move[5]

                    board[future_move[0]][future_move[1]].occupied = False
                    board[future_move[0]][future_move[1]].piece = None
                    board[future_move[0]][future_move[1]].piece_color = None
                elif future_move[-1] == 'castling droit':
                    #pour le roi
                    board[future_move[2]][future_move[3]].occupied = True
                    board[future_move[2]][future_move[3]].piece = future_move[6]
                    board[future_move[2]][future_move[3]].piece_color = future_move[5]

                    board[future_move[0]][future_move[1]].occupied = False
                    board[future_move[0]][future_move[1]].piece = None
                    board[future_move[0]][future_move[1]].piece_color = None
                    
                    #pour la tour
                    board[future_move[2]][future_move[3]-1].occupied = True
                    board[future_move[2]][future_move[3]-1].piece = 'rook'
                    board[future_move[2]][future_move[3]-1].piece_color = future_move[5]

                    board[future_move[2]][7].occupied = False
                    board[future_move[2]][7].piece = None
                    board[future_move[2]][7].piece_color = None
                  
                elif future_move[-1] == 'castling gauche':
                    #pour le roi
                    board[future_move[2]][future_move[3]].occupied = True
                    board[future_move[2]][future_move[3]].piece = future_move[6]
                    board[future_move[2]][future_move[3]].piece_color = future_move[5]

                    board[future_move[0]][future_move[1]].occupied = False
                    board[future_move[0]][future_move[1]].piece = None
                    board[future_move[0]][future_move[1]].piece_color = None
                    
                    #pour la tour
                    board[future_move[2]][future_move[3]+1].occupied = True
                    board[future_move[2]][future_move[3]+1].piece = 'rook'
                    board[future_move[2]][future_move[3]+1].piece_color = future_move[5]

                    board[future_move[2]][0].occupied = False
                    board[future_move[2]][0].piece = None
                    board[future_move[2]][0].piece_color = None
                else:
                    board[future_move[2]][future_move[3]].occupied = True
                    board[future_move[2]][future_move[3]].piece = future_move[6]
                    board[future_move[2]][future_move[3]].piece_color = future_move[5]

                    board[future_move[0]][future_move[1]].occupied = False
                    board[future_move[0]][future_move[1]].piece = None
                    board[future_move[0]][future_move[1]].piece_color = None
                    
                
                







                 

SystemExit: 

In [5]:
all_last_move

[(6, 1, 5, 1, None, 'white', 'wpawn', None),
 (1, 6, 2, 6, None, 'black', 'bpawn', None),
 (6, 5, 5, 5, None, 'white', 'wpawn', None),
 (1, 5, 3, 5, None, 'black', 'bpawn', None),
 (6, 6, 4, 6, None, 'white', 'wpawn', None),
 (0, 1, 2, 0, None, 'black', 'knight', None),
 (7, 5, 6, 6, None, 'white', 'bishop', None),
 (1, 1, 3, 1, None, 'black', 'bpawn', None),
 (7, 1, 5, 2, None, 'white', 'knight', None),
 (1, 3, 2, 3, None, 'black', 'bpawn', None),
 (6, 7, 5, 7, None, 'white', 'wpawn', None),
 (0, 5, 1, 6, None, 'black', 'bishop', None),
 (7, 0, 7, 1, None, 'white', 'rook', None),
 (2, 6, 3, 6, None, 'black', 'bpawn', None),
 (5, 5, 4, 5, None, 'white', 'wpawn', None),
 (3, 6, 4, 5, 'wpawn', 'black', 'bpawn', None),
 (4, 6, 3, 5, 'bpawn', 'white', 'wpawn', None),
 (1, 7, 2, 7, None, 'black', 'bpawn', None),
 (7, 4, 6, 5, None, 'white', 'king', None),
 (1, 4, 2, 4, None, 'black', 'bpawn', None)]

In [ ]:
'''
deboguage
-Les pieces sont pinnees meme si la piece adverse menacante se trouve derriere une piece adverse non menacante: RÉSOLU erreur dans les if de show_move() pas dans pinned()
-Les noirs ne sont pas obligés de ne pas bouger avec la menace des blancs: RESOLU self.pinned se faisait reinitialiser apres s'etre fait attribuer une valeur. erreur d'ordre
-Castling mm si le roi va pas dans la direction voulue: RESOLU le castling dans self.move et dans self.show_move était fait tout croche jai additionne les strings a la place de les ecraser
-Quand il n'y a aucun mouv possible pour une couleur, il ne se passe rien: RESOLU Mtn si une couleur ne peut pas faire de mouvement, elle est en echec et mat
-Apres une mise en echec proche des bords, la case retrouve son etat False pour l'attribut self.piece_moved somehow: RESOLU a chaque fois qu<on regarde le board on appel case et c<est pas bon. j'ai change case pour carre. pas d'erreur
-Apres un echec et mat, les robots envoient toujours des informations?: RESOLU je les fais arrêter jusqu'a ce que je jeu sois restarté avec r
-Pinned() fonctionne pas tout le temps: RESOLU pinned est séparé de show_move(self), je l'ai mis dedans 
-Pinned() pinnait des piéces alors que le roi n'était pas menacé: RÉSOLU la direction d'attaque des piecemenacantes n'était pas prise en compte. mtn elle l'est
-Effet incertain de l'echec a certains moments... à vérifier 07-02
-Le roi s'est fait manger par un robot... trouver pourquoi 07-02
-Les pions sur le bord du jeu regardent a l'exterieur du jeu : RESOLU je les ai empêché de le faire NON je viens de le revoir 07-02
-Cheval met le roir en echec et mat meme si il peut se faire manger

Extras
restart rKEY FAIT
mettre option 2p et option 1p et 0p FAIT
ctrl z et y pour 1 mouv mais incluant les mouvs speciaux (castling(FAIT), en passant(FAIT), pion devient reine(FAIT))
Rendre BATO() plus agressive en la faisant manger si elle en a la chance FAIT
ctrl z et y pour tt les mouvs FAIT
Se souvenir de tous les mouvements FAIT
Empecher le castling si le roi a ete en echec
Comptabiliser les points
Montrer les pièces mangées
'''